# 1.5 — Pandas
**DataFrames, Cleaning, EDA**

> Core idea: Pandas is your data workbench. Before any model touches the data, you use Pandas to load, inspect, clean, and shape it.

## Why Pandas?

NumPy is fast but it doesn't know about column names, mixed data types, or missing values. 

Real datasets have:
- Columns with names like `age`, `salary`, `city`
- Mixed types — numbers AND text in the same table
- Missing values (NaN)
- Dates, categories, messy formatting

Pandas handles all of this. It's built on NumPy but adds the intelligence of a spreadsheet.

## Analogy — NumPy vs Pandas

NumPy = a grid of raw numbers. Fast, no labels.
Pandas = an Excel sheet. Named columns, mixed types, easy filtering.

In [1]:
import pandas as pd
import numpy as np

# Creating a DataFrame
data = {
    'name':   ['Arwindh', 'Priya', 'Ravi', 'Meena', 'Kumar'],
    'age':    [24, 28, 35, 22, 45],
    'salary': [35000, 55000, 80000, 30000, None],    # None = missing
    'city':   ['Kochi', 'Chennai', 'Bangalore', 'Kochi', 'Chennai'],
    'loan':   [1, 0, 0, 1, 1]                        # 1 = took loan
}

df = pd.DataFrame(data)
print(df)

      name  age   salary       city  loan
0  Arwindh   24  35000.0      Kochi     1
1    Priya   28  55000.0    Chennai     0
2     Ravi   35  80000.0  Bangalore     0
3    Meena   22  30000.0      Kochi     1
4    Kumar   45      NaN    Chennai     1


In [2]:
# --- FIRST THING YOU DO ON ANY NEW DATASET ---
print("Shape:", df.shape)              # (rows, cols)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic stats:")
print(df.describe())

Shape: (5, 5)

Column types:
name          str
age         int64
salary    float64
city          str
loan        int64
dtype: object

Missing values:
name      0
age       0
salary    1
city      0
loan      0
dtype: int64

Basic stats:
             age        salary      loan
count   5.000000      4.000000  5.000000
mean   30.800000  50000.000000  0.600000
std     9.364828  22730.302828  0.547723
min    22.000000  30000.000000  0.000000
25%    24.000000  33750.000000  0.000000
50%    28.000000  45000.000000  1.000000
75%    35.000000  61250.000000  1.000000
max    45.000000  80000.000000  1.000000


In [3]:
# Selecting data
print("Single column (Series):")
print(df['age'])

print("\nMultiple columns (DataFrame):")
print(df[['name', 'salary']])

print("\nRow by index:")
print(df.iloc[0])    # first row by position

print("\nRow by label:")
print(df.loc[2])     # row with index label 2

Single column (Series):
0    24
1    28
2    35
3    22
4    45
Name: age, dtype: int64

Multiple columns (DataFrame):
      name   salary
0  Arwindh  35000.0
1    Priya  55000.0
2     Ravi  80000.0
3    Meena  30000.0
4    Kumar      NaN

Row by index:
name      Arwindh
age            24
salary    35000.0
city        Kochi
loan            1
Name: 0, dtype: object

Row by label:
name           Ravi
age              35
salary      80000.0
city      Bangalore
loan              0
Name: 2, dtype: object


In [ ]:
# Filtering rows
kochi_people = df[df['city'] == 'Kochi']
print("Kochi residents:\n", kochi_people)

young_earners = df[(df['age'] < 30) & (df['salary'] > 40000)]
print("\nYoung high earners:\n", young_earners)

In [ ]:
# Handling missing values
print("Before:\n", df['salary'])

df['salary'] = df['salary'].fillna(df['salary'].median())
print("\nAfter (filled with median):\n", df['salary'])

In [4]:
# Creating new columns (feature engineering)
df['salary_in_lakhs'] = df['salary'] / 100000
df['is_high_earner'] = (df['salary'] > 50000).astype(int)
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 100], labels=['young', 'mid', 'senior'])

print(df[['name', 'salary', 'salary_in_lakhs', 'is_high_earner', 'age_group']])

      name   salary  salary_in_lakhs  is_high_earner age_group
0  Arwindh  35000.0             0.35               0     young
1    Priya  55000.0             0.55               1       mid
2     Ravi  80000.0             0.80               1       mid
3    Meena  30000.0             0.30               0     young
4    Kumar      NaN              NaN               0    senior


In [5]:
# Groupby — aggregate by category
print("Avg salary by city:")
print(df.groupby('city')['salary'].mean())

print("\nLoan rate by age group:")
print(df.groupby('age_group')['loan'].mean())

Avg salary by city:
city
Bangalore    80000.0
Chennai      55000.0
Kochi        32500.0
Name: salary, dtype: float64

Loan rate by age group:
age_group
young     1.0
mid       0.0
senior    1.0
Name: loan, dtype: float64


In [6]:
# Encoding for ML — must convert text to numbers
# One-hot encoding (no natural order)
df_encoded = pd.get_dummies(df, columns=['city'], drop_first=True)
print("After one-hot encoding:\n", df_encoded.head())

After one-hot encoding:
       name  age   salary  loan  salary_in_lakhs  is_high_earner age_group  \
0  Arwindh   24  35000.0     1             0.35               0     young   
1    Priya   28  55000.0     0             0.55               1       mid   
2     Ravi   35  80000.0     0             0.80               1       mid   
3    Meena   22  30000.0     1             0.30               0     young   
4    Kumar   45      NaN     1              NaN               0    senior   

   city_Chennai  city_Kochi  
0         False        True  
1          True       False  
2         False       False  
3         False        True  
4          True       False  


In [7]:
# --- SPLITTING INTO X AND y ---
# This is the final step before sklearn

# Target = 'loan' column
X = df_encoded.drop(columns=['name', 'loan', 'age_group', 'salary_in_lakhs', 'is_high_earner'])
y = df_encoded['loan']

print("X shape:", X.shape)   # 2D — features
print("y shape:", y.shape)   # 1D — target
print("\nX:\n", X)
print("\ny:", y.values)

X shape: (5, 4)
y shape: (5,)

X:
    age   salary  city_Chennai  city_Kochi
0   24  35000.0         False        True
1   28  55000.0          True       False
2   35  80000.0         False       False
3   22  30000.0         False        True
4   45      NaN          True       False

y: [1 0 0 1 1]


## Pandas Quick Reference for ML

```python
# Load
df = pd.read_csv('file.csv')

# Inspect
df.shape          # dimensions
df.head()         # first 5 rows
df.info()         # types + non-null counts
df.describe()     # stats summary
df.isnull().sum() # missing counts
df.dtypes         # data types per column

# Clean
df.fillna(value)            # fill missing
df.dropna()                 # drop missing rows
df.drop(columns=['col'])    # remove column
df.rename(columns={'old': 'new'})

# Transform
df['new'] = df['a'] / df['b']     # new feature
pd.get_dummies(df, columns=['c']) # one-hot encode

# Prepare for ML
X = df.drop(columns=['target'])
y = df['target']
```

## Practice Task

Load the Titanic dataset and run the standard first inspection.

In [ ]:
import seaborn as sns
titanic = sns.load_dataset('titanic')

# YOUR CODE HERE
# Q1: What is the shape of the dataset?

# Q2: Which columns have missing values and how many?

# Q3: What is the survival rate overall? (mean of 'survived' column)

# Q4: What is the average age of survivors vs non-survivors?
# Hint: titanic.groupby('survived')['age'].mean()

# Q5: Create X (drop 'survived') and y ('survived' column)